<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DeepLearning/blob/main/LLM_Experimente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DeepLearning/blob/main/LLM_BusinessAI_Colab_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Large Language Models: Von Tokens zu Antworten

In diesem Notebook schaust du einem Sprachmodell beim Arbeiten zu – Schritt für Schritt,
vom einzelnen Token bis zur fertigen Antwort.

## Lernziele

Nach diesem Notebook kannst du:

1. **Tokenisierung** erklären und begründen, warum deutsche Texte in der API mehr kosten als englische.
2. Die **Wahrscheinlichkeitsverteilung** hinter jedem generierten Token sichtbar machen.
3. Erklären, was **Temperature**, **Top-k** und **Top-p** mit dieser Verteilung machen.
4. Den Effekt des **KV-Caches** messen.
5. Den Unterschied zwischen einem **Basismodell** und einem **Assistenzmodell** (Chat-Template, System-Prompt) zeigen.
6. Beurteilen, wann LLM-Ausgaben im Geschäftskontext **belastbar** sind und wann nicht.

## Praktisches

- Läuft in Google Colab **ohne GPU**. Mit GPU (`Laufzeit` → `Laufzeittyp ändern` → `T4 GPU`) sind Teil 4 und 5 deutlich schneller.
- Gesamtlaufzeit auf CPU: ca. 5–8 Minuten, der grösste Teil davon sind die Modell-Downloads.
- Führe die Zellen **von oben nach unten** aus.

> Die Aufgaben am Ende jedes Teils folgen dem Schema **Hypothese → Experiment → Resultat → Interpretation**.
> Schreib deine Hypothese auf, *bevor* du die Zelle laufen lässt. Falsche Hypothesen sind die lehrreichen.

## Setup

In [ ]:
!pip install -q -U transformers

import time
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging

# Fortschrittsbalken ausblenden, damit die Ausgaben lesbar bleiben
hf_logging.disable_progress_bar()
hf_logging.set_verbosity_error()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Einheitliche Farben für alle Grafiken in diesem Notebook
C_BLAU, C_ORANGE, C_GRUEN = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11})

---
# Teil 1: Tokenisierung

Ein Sprachmodell sieht keinen Text. Es sieht **Tokens** – ganze Zahlen, die für Wortteile stehen.
Alles, was du bezahlst (API-Kosten), und alles, was ins Kontextfenster passt, wird in Tokens gemessen.

## 1.1 Vom Text zum Token

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Explain market segmentation in one sentence."
ids = tokenizer(text)["input_ids"]

print("Text:  ", text)
print("IDs:   ", ids)
print("Tokens:", [tokenizer.decode([i]) for i in ids])
print(f"\nAnzahl Tokens: {len(ids)}")

Beachte die führenden Leerzeichen in den Tokens (`' market'`). Der Tokenizer kodiert das Leerzeichen mit –
deshalb sind `"Markt"` und `" Markt"` zwei verschiedene Tokens.

## 1.2 Warum Deutsch teurer ist

GPT-2 wurde fast ausschliesslich auf englischem Text trainiert. Sein Vokabular kennt englische Wörter als Ganzes,
deutsche muss es zusammenstückeln.

In [ ]:
satz_de = "Die Kundensegmentierung ermöglicht eine zielgruppengenaue Ansprache für Grossunternehmen."
satz_en = "Customer segmentation enables precise targeting for large enterprises."

for label, satz in [("DE", satz_de), ("EN", satz_en)]:
    t_ids = tokenizer(satz)["input_ids"]
    print(f"{label}: {len(t_ids):3d} Tokens für {len(satz.split()):2d} Wörter  ({len(satz)} Zeichen)")

print("\nSo zerlegt GPT-2 den deutschen Satz:")
print([tokenizer.decode([i]) for i in tokenizer(satz_de)["input_ids"]])

## 1.3 Haben moderne Tokenizer das Problem gelöst?

Moderne Modelle werden mehrsprachig trainiert und haben ein grösseres Vokabular. Vergleichen wir.

In [ ]:
modelle = {
    "GPT-2 (2019)": "gpt2",
    "Qwen3 (2025)": "Qwen/Qwen3-0.6B",
}

ergebnisse = {}
for name, pfad in modelle.items():
    tok = AutoTokenizer.from_pretrained(pfad)
    n_de = len(tok(satz_de)["input_ids"])
    n_en = len(tok(satz_en)["input_ids"])
    ergebnisse[name] = (n_de, n_en)
    print(f"{name:14s}  DE: {n_de:3d}   EN: {n_en:3d}   Faktor: {n_de/n_en:.2f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))

namen = list(ergebnisse.keys())
x = range(len(namen))
breite = 0.34
de_werte = [ergebnisse[n][0] for n in namen]
en_werte = [ergebnisse[n][1] for n in namen]

b1 = ax.bar([i - breite/2 - 0.025 for i in x], de_werte, breite, label="Deutsch", color=C_ORANGE)
b2 = ax.bar([i + breite/2 + 0.025 for i in x], en_werte, breite, label="Englisch", color=C_BLAU)

for balken in (b1, b2):
    for rect in balken:
        ax.text(rect.get_x() + rect.get_width()/2, rect.get_height() + 0.6,
                f"{int(rect.get_height())}", ha="center", va="bottom", fontsize=10)

ax.set_xticks(list(x)); ax.set_xticklabels(namen)
ax.set_ylabel("Tokens für denselben Satz")
ax.set_title("Dieselbe Aussage, zwei Sprachen", loc="left", fontsize=12)
ax.set_ylim(0, max(de_werte) * 1.2)
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

### Aufgabe 1

**Hypothese:** Ein Kundenservice-Bot verarbeitet täglich 10'000 Anfragen. Du rechnest mit 150 Tokens pro Anfrage
auf Englisch. Wie viel teurer wird der Betrieb auf Deutsch? Schreib deine Schätzung auf, bevor du rechnest.

**Experiment:** Nimm den Faktor aus 1.3 und rechne die Jahreskosten für beide Sprachen aus.
Rechne mit 0.50 CHF pro 1 Mio. Input-Tokens.

**Resultat & Interpretation:** Welche Konsequenz hat das für die Modellwahl eines Schweizer KMU?
Nenne mindestens einen Fall, in dem der Tokenizer das entscheidende Auswahlkriterium ist.

In [ ]:
# Platz für deine Rechnung
anfragen_pro_tag = 10_000
tokens_en = 150
preis_pro_mio = 0.50

# TODO: Faktor aus 1.3 einsetzen und Jahreskosten für DE und EN berechnen

---
# Teil 2: Generierung mit einem Basismodell

Wir laden GPT-2. Das Modell ist **nicht instruktionsgetunt** – es setzt Text fort, es beantwortet keine Fragen.
Genau deshalb eignet es sich, um die Mechanik zu verstehen: Was du siehst, ist reine Fortsetzung, ohne Assistenz-Training.

## 2.1 Erste Generierung

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
model.eval()

prompt = "Explain deep learning in simple terms:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

torch.manual_seed(42)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=40, do_sample=True,
                         temperature=0.7, top_k=50,
                         pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(out[0], skip_special_tokens=True))

## 2.2 Die Verteilung hinter jedem Token

Das Modell wählt nicht "ein Wort". Es berechnet für **jedes** der über 50'000 Tokens im Vokabular eine Wahrscheinlichkeit.
Schauen wir uns diese Verteilung einmal direkt an – ein einziger Forward-Pass, ohne `generate()`.

In [ ]:
satzanfang = "The customer is always"
inp = tokenizer(satzanfang, return_tensors="pt").to(device)

with torch.no_grad():
    logits = model(**inp).logits[0, -1]   # Logits für das NÄCHSTE Token

wahrscheinlichkeiten = torch.softmax(logits, dim=-1)
top = torch.topk(wahrscheinlichkeiten, 10)

print(f"Vokabulargrösse: {len(wahrscheinlichkeiten)} Tokens")
print(f"Fortsetzung von: '{satzanfang}'\n")
for wert, idx in zip(top.values, top.indices):
    print(f"  {tokenizer.decode([idx]):>12s}  {wert.item()*100:5.1f}%")

## 2.3 Was Temperature mit dieser Verteilung macht

Temperature teilt die Logits, *bevor* Softmax angewendet wird: `softmax(logits / T)`.

- **T < 1** spreizt die Verteilung: Wahrscheinliches wird noch wahrscheinlicher.
- **T > 1** flacht sie ab: Unwahrscheinliches bekommt eine echte Chance.
- **T → 0** entspricht Greedy Decoding (immer das wahrscheinlichste Token).

In [ ]:
temperaturen = [0.2, 1.0, 2.0]
farben = [C_BLAU, C_ORANGE, C_GRUEN]

# Die 6 wahrscheinlichsten Kandidaten bei T=1.0 als gemeinsame Basis
basis = torch.topk(torch.softmax(logits, dim=-1), 6).indices
labels = [tokenizer.decode([i]).strip() or "_" for i in basis]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)

for ax, T, farbe in zip(axes, temperaturen, farben):
    p = torch.softmax(logits / T, dim=-1)[basis]
    balken = ax.bar(labels, (p * 100).cpu().numpy(), color=farbe, width=0.68)
    for rect in balken:
        ax.text(rect.get_x() + rect.get_width()/2, rect.get_height() + 1.0,
                f"{rect.get_height():.1f}", ha="center", va="bottom", fontsize=9)
    ax.set_title(f"T = {T}", loc="left", fontsize=12)
    ax.tick_params(axis="x", rotation=45)

# Kopfraum, damit die Beschriftungen nicht in die Titel laufen
max_wert = torch.softmax(logits / min(temperaturen), dim=-1)[basis].max().item() * 100
axes[0].set_ylim(0, max_wert * 1.3)
axes[0].set_ylabel("Wahrscheinlichkeit (%)")
fig.suptitle(f"Nächstes Token nach '{satzanfang}'", x=0.01, ha="left", fontsize=13)
plt.tight_layout(); plt.show()

Beachte die Achse: Bei `T = 2.0` sind alle Balken fast gleich hoch. Das Modell "weiss" immer noch dasselbe –
wir entscheiden nur, wie stark wir sein Wissen beim Ziehen gewichten.

## 2.4 Sampling-Strategien im kontrollierten Vergleich

**Wichtig:** Ohne festen Zufallsstartwert unterscheiden sich zwei Generierungen durch die Einstellung *und* durch Zufall.
Dann lässt sich der Effekt nicht zuordnen. Wir setzen deshalb vor jedem Aufruf denselben Seed.

In [ ]:
def generiere(prompt, seed=42, max_new_tokens=35, **kwargs):
    """Generiert Text mit festem Zufallsstartwert, damit Einstellungen vergleichbar sind."""
    inp = tokenizer(prompt, return_tensors="pt").to(device)
    torch.manual_seed(seed)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens,
                             pad_token_id=tokenizer.eos_token_id, **kwargs)
    return tokenizer.decode(out[0], skip_special_tokens=True)


prompt = "The main advantage of cloud computing for small businesses is"

strategien = {
    "Greedy (immer Platz 1)": dict(do_sample=False),
    "T=0.2 (konservativ)":    dict(do_sample=True, temperature=0.2, top_k=0),
    "T=1.5 (kreativ)":        dict(do_sample=True, temperature=1.5, top_k=0),
    "Top-k = 10":             dict(do_sample=True, temperature=1.0, top_k=10),
    "Top-p = 0.9 (Nucleus)":  dict(do_sample=True, temperature=1.0, top_k=0, top_p=0.9),
}

for name, einstellung in strategien.items():
    print(f"--- {name} ---")
    print(generiere(prompt, **einstellung))
    print()

**Top-k vs. Top-p:** Top-k nimmt immer die *k* wahrscheinlichsten Tokens – unabhängig davon, wie die Verteilung aussieht.
Top-p (Nucleus) nimmt so viele Tokens, bis deren Wahrscheinlichkeiten zusammen *p* ergeben. Die Menge passt sich also an:
Bei einer eindeutigen Fortsetzung bleiben wenige Kandidaten übrig, bei einer offenen viele.

### Aufgabe 2

**Hypothese:** Du baust einen Generator für Produktbeschreibungen in einem Online-Shop.
Welche Einstellung wählst du – und warum? Welche wählst du für die Extraktion von Rechnungsdaten?

**Experiment:** Setze in der Funktion `generiere()` den Seed auf einen anderen Wert und lass Zelle 2.4 erneut laufen.

**Resultat:** Welche Strategie liefert bei unterschiedlichen Seeds identischen Output – und welche nicht?

**Interpretation:** Was bedeutet das für die **Reproduzierbarkeit** eines Systems, das du einem Kunden ausliefern musst?

---
# Teil 3: Der KV-Cache

Bei jedem neuen Token müsste das Modell eigentlich die gesamte bisherige Sequenz erneut durchrechnen.
Der **KV-Cache** speichert die Keys und Values der bereits verarbeiteten Tokens, sodass pro Schritt nur noch
ein neues Query berechnet werden muss.

Messen wir den Unterschied.

In [ ]:
prompt = "The quarterly report shows that"
inp = tokenizer(prompt, return_tensors="pt").to(device)

zeiten = {}
for use_cache in [True, False]:
    start = time.time()
    with torch.no_grad():
        model.generate(**inp, max_new_tokens=100, do_sample=False,
                       use_cache=use_cache, pad_token_id=tokenizer.eos_token_id)
    zeiten[use_cache] = time.time() - start
    print(f"use_cache={str(use_cache):5s}: {zeiten[use_cache]:5.2f}s für 100 Tokens")

print(f"\nBeschleunigung: {zeiten[False] / zeiten[True]:.1f}x")

### Aufgabe 3

**Hypothese:** Wie verändert sich dieser Faktor, wenn du statt 100 nur 20 Tokens generierst? Und bei 300?

**Experiment:** Ändere `max_new_tokens` und miss erneut.

**Interpretation:** Der Aufwand ohne Cache wächst quadratisch mit der Sequenzlänge, mit Cache linear.
Warum ist das der Grund, weshalb lange Kontextfenster teuer sind – und warum Anbieter "Prompt Caching" anbieten?

---
# Teil 4: Vom Basismodell zum Assistenten

GPT-2 kann Text fortsetzen – mehr nicht. Alles, was du von ChatGPT oder Claude kennst
(Anweisungen befolgen, Rollen einnehmen, strukturiert antworten), kommt aus dem **Finetuning** danach.

Wir laden ein kleines, modernes **Assistenzmodell**: Qwen3-0.6B (Apache-2.0-Lizenz, läuft auf CPU).

## 4.1 Derselbe Prompt, zwei Modelle

In [ ]:
chat_name = "Qwen/Qwen3-0.6B"
chat_tok = AutoTokenizer.from_pretrained(chat_name)
chat_model = AutoModelForCausalLM.from_pretrained(
    chat_name,
    dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
chat_model.eval()


def frage(nachricht, system="Du bist ein hilfreicher Assistent.", seed=42,
          max_new_tokens=200, temperature=0.7):
    """Stellt dem Assistenzmodell eine Frage über das Chat-Template."""
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": nachricht}]
    text = chat_tok.apply_chat_template(msgs, tokenize=False,
                                        add_generation_prompt=True,
                                        enable_thinking=False)
    inp = chat_tok(text, return_tensors="pt").to(device)
    torch.manual_seed(seed)
    with torch.no_grad():
        out = chat_model.generate(**inp, max_new_tokens=max_new_tokens,
                                  do_sample=True, temperature=temperature, top_p=0.9,
                                  pad_token_id=chat_tok.eos_token_id)
    return chat_tok.decode(out[0][inp["input_ids"].shape[1]:],
                           skip_special_tokens=True).strip()


aufgabe = "Nenne drei Vorteile von Cloud Computing für KMU."

print("=== GPT-2 (Basismodell) ===")
print(generiere(aufgabe, do_sample=True, temperature=0.7, top_k=50, max_new_tokens=200))
print("\n=== Qwen3-0.6B (Assistenzmodell) ===")
print(frage(aufgabe))

Derselbe Prompt, zwei völlig verschiedene Verhalten. Der Unterschied liegt **nicht** in der Architektur –
beides sind Decoder-Transformer. Er liegt im Training danach.

> **Zur Einordnung:** Qwen3-0.6B hat 0.6 Milliarden Parameter – ein Bruchteil eines aktuellen Spitzenmodells.
> Sein Deutsch ist stellenweise holprig, gelegentlich rutscht ein fremdsprachiges Wort hinein. Das ist erwartbar
> und ändert nichts an den Mechanismen, um die es hier geht.

## 4.2 Was das Chat-Template wirklich macht

"Rollen" gibt es im Modell nicht. Es gibt nur eine Token-Sequenz. Das Chat-Template ist die Vorschrift,
wie System-, User- und Assistenz-Nachrichten in **einen** Text gegossen werden. Schauen wir ihn uns an.

In [ ]:
msgs = [
    {"role": "system", "content": "Du bist ein Analyst und antwortest in Stichworten."},
    {"role": "user", "content": "Was ist Kundenabwanderung?"},
]

print(chat_tok.apply_chat_template(msgs, tokenize=False,
                                   add_generation_prompt=True,
                                   enable_thinking=False))

Die Sonderzeichen (`<|im_start|>`, `<|im_end|>`) sind eigene Tokens, die das Modell im Finetuning gelernt hat.
Das ist der ganze Trick hinter "Rollen".

Der leere `<think></think>`-Block stammt von `enable_thinking=False`: Qwen3 kann vor der Antwort einen
Denkschritt ausgeben. Wir schalten ihn ab, damit die Ausgabe kurz bleibt.

> **Sicherheitsrelevant:** Was passiert, wenn ein User diese Sonderzeichen selbst in seine Nachricht schreibt?
> Das ist der Kern von **Prompt Injection**.

## 4.3 Der System-Prompt steuert das Verhalten

In [ ]:
nachricht = "Was bringt einem KMU der Wechsel in die Cloud?"

system_prompts = {
    "Stichworte": ("Du antwortest ausschliesslich mit genau drei Stichworten. "
                   "Keine ganzen Sätze, keine Erklärungen."),
    "Risiko-Berater": ("Du bist ein vorsichtiger IT-Berater. "
                       "Du nennst zuerst die Risiken, danach erst die Vorteile."),
}

for name, sp in system_prompts.items():
    print(f"--- System-Prompt: {name} ---")
    print(frage(nachricht, system=sp, max_new_tokens=80, temperature=0.6))
    print()

## 4.4 In-Context Learning: Lernen ohne Training

Beispiele im Prompt verändern **keine** Modellgewichte. Sie verändern nur den Kontext –
und trotzdem ändert sich das Verhalten. Das ist **In-Context Learning**.

In [ ]:
zero_shot = "Klassifiziere die Stimmung: 'Die Lieferung kam drei Tage zu spät.'"

few_shot = """Klassifiziere die Stimmung als POSITIV, NEUTRAL oder NEGATIV.

Text: "Super Produkt, gerne wieder!"
Label: POSITIV

Text: "Die Rechnung liegt bei."
Label: NEUTRAL

Text: "Die Lieferung kam drei Tage zu spät."
Label:"""

print("=== Zero-Shot ===")
print(frage(zero_shot, max_new_tokens=60))
print("\n=== Few-Shot ===")
print(frage(few_shot, max_new_tokens=20))

### Aufgabe 4

**Hypothese:** Few-Shot-Beispiele kosten Tokens in *jeder* Anfrage. Ab wann lohnt sich stattdessen ein Finetuning?

**Experiment:** Baue einen Few-Shot-Prompt für eine Klassifikationsaufgabe aus deinem Projekt
(z.B. Ticket-Kategorie, Risikostufe). Teste ihn mit drei verschiedenen Eingaben.
Zähle mit `chat_tok()`, wie viele Tokens dein Prompt kostet.

**Resultat:** Wie zuverlässig ist das Ausgabeformat? Hält sich das Modell immer an deine Labels?

**Interpretation:** Welche Absicherung brauchst du im Produktivsystem, wenn das Ausgabeformat nicht garantiert ist?

---
# Teil 5: Wann sind LLM-Ausgaben belastbar?

Ein Sprachmodell optimiert auf **plausible Fortsetzung**, nicht auf Wahrheit.
Eine erfundene Zahl ist für das Modell genauso "korrekt" wie eine richtige – sie passt ins Muster.

## 5.1 Eine Frage, auf die es keine Antwort geben kann

Die folgende Firma existiert nicht. Ein ehrliches System müsste das sagen – achte darauf, was stattdessen
passiert, und wie **selbstsicher** die Antworten formuliert sind.

In [ ]:
frage_text = ("In welchem Jahr wurde die Nordwind Analytics AG gegründet "
              "und wer war der Gründer?")

for i, seed in enumerate([1, 2, 3], start=1):
    print(f"--- Antwort {i} (Seed {seed}) ---")
    print(frage(frage_text, seed=seed, max_new_tokens=70, temperature=0.9))
    print()

## 5.2 Selbstkonsistenz als Warnsignal

Vergleiche die drei Antworten: Wo das Modell etwas **weiss**, bleiben die Antworten über verschiedene Seeds hinweg stabil.
Wo es **konfabuliert**, widersprechen sie sich. Das ist ein billiges, praktisch nutzbares Prüfsignal –
mehrfaches Sampeln und Vergleichen deckt einen Teil der Halluzinationen auf.

Zum Gegentest eine Frage, deren Antwort im Trainingsmaterial breit vorkommt:

In [ ]:
kontrollfrage = "Was bedeutet die Abkürzung CRM im Marketing? Antworte in einem Satz."

for i, seed in enumerate([1, 2, 3], start=1):
    print(f"--- Antwort {i} (Seed {seed}) ---")
    print(frage(kontrollfrage, seed=seed, max_new_tokens=50, temperature=0.9))
    print()

### Aufgabe 5

**Hypothese:** Bei welcher der beiden Fragen erwartest du mehr Streuung zwischen den Seeds? Warum?

**Experiment:** Formuliere zwei eigene Fragen aus deinem Projektkontext – eine nachprüfbare Faktenfrage
und eine, deren Antwort das Modell nicht kennen kann. Lass beide dreimal mit verschiedenen Seeds laufen.

**Resultat:** Dokumentiere, wo die Antworten auseinanderlaufen.

**Interpretation:** Für welche Aufgaben im Unternehmen würdest du ein LLM **ohne** menschliche Kontrolle einsetzen –
und für welche nie? Begründe anhand von zwei Kriterien: Kosten eines Fehlers und Prüfbarkeit der Ausgabe.

> **Hinweis zur Modellgrösse:** Qwen3-0.6B ist winzig. Grosse Modelle halluzinieren seltener und erkennen eher,
> wenn sie etwas nicht wissen – aber das Grundproblem bleibt dasselbe.

---
# Zusammenfassung

| Konzept | Was du gesehen hast |
|---|---|
| Tokenisierung | Deutsch kostet bei GPT-2 ein Vielfaches von Englisch; moderne Tokenizer reduzieren das |
| Verteilung | Hinter jedem Token steht eine Wahrscheinlichkeit über das ganze Vokabular |
| Temperature | Skaliert die Logits vor Softmax – spreizt oder flacht die Verteilung ab |
| Top-k / Top-p | Begrenzen die Kandidatenmenge fix bzw. adaptiv |
| KV-Cache | Macht den Aufwand pro Token linear statt quadratisch |
| Chat-Template | "Rollen" sind Sonderzeichen in einer einzigen Token-Sequenz |
| In-Context Learning | Beispiele im Prompt ändern das Verhalten ohne Gewichtsänderung |
| Halluzination | plausibel heisst nicht automatisch korrekt; Selbstkonsistenz ist ein Prüfsignal |